# TO DO

- filtrar apenas colunas sem formato degrau

- Cortar o intervalo de tempo de 0 a 1s

- Ajustar o time interval para uma frequencia de 100HZ (de 0.01s em 0.01s)

# Preparando os dados

- O objetivo deste projeto é detectar anomalias no motor da aeronave o mais rápido possível

- Isso implica que a anomalia já tenha ocorrido e a intenção do modelo é detectar esta anomalia o quanto antes para emitir um sinal de alerta.

- A detecção preditiva, ou seja, antes que a falha ocorra não é possível com os dados do ALFA dataset, dado que a falha é induzida por um sinal abrupto externo que desliga o motor, e não por falhas mecânicas gradativas que indicariam uma possível futura.



- Links uteis:

1 - https://theairlab.org/alfa-dataset/ (contém informações sobre o status do vôo.)

# Explorando os dados


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Parâmetros

In [ ]:
DATA_FILE = '../data/02_intermediate/carbonZ_2018-07-18-15-53-31_1_engine_failure_pre_processed.csv'

# Funções

In [ ]:
def plot_flight_data_by_source(df):
    # 1. Filtrar as colunas: removemos o 'timestamp' e o 'temp_index' (se existir)
    # Assim, o número de subplots será exatamente igual ao número de variáveis
    cols_to_plot = [c for c in df.columns if c not in ['timestamp', 'temp_index', '%time']]
    
    n_vars = len(cols_to_plot)
    
    if n_vars == 0:
        print("Nenhuma coluna encontrada para plotar.")
        return

    # 2. Configurar o layout baseado apenas nas colunas de dados
    fig, axes = plt.subplots(n_vars, 1, figsize=(15, 2.5 * n_vars), sharex=True)
    
    # Garantir que axes seja sempre uma lista (mesmo com 1 subplot)
    if n_vars == 1:
        axes = [axes]

    # 3. Gerar os plots usando a lista filtrada
    for i, col in enumerate(cols_to_plot):
        # Tenta limpar o nome para o label (removendo prefixos chatos)
        clean_label = col.split('.')[-1] if '.' in col else col
        
        axes[i].plot(df['timestamp'], df[col], label=col, color='tab:blue')
        
        # Destacar o momento da falha (aproximadamente 118s no seu caso)
        axes[i].axvline(x=118, color='red', linestyle='--', alpha=0.6, label='Fault Injection')
        
        axes[i].set_ylabel(clean_label)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(loc='upper right', fontsize='small')

    # Ajuste final do eixo X
    axes[-1].set_xlabel('Tempo (segundos)')
    
    plt.suptitle('Análise Temporal dos Sensores de Voo', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    plt.show()

# Para usar, basta chamar:
# plot_flight_data_by_source(df5)

# Abrindo todos os arquivos e salvando em um dicionário de dfs



In [ ]:
df = pd.read_csv(DATA_FILE)

In [ ]:
df.columns[80:]

In [ ]:

# Dicionário de mapeamento: "Nome Antigo": "Nome Novo"
rename_dict = {
    '%time': 'timestamp',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-failure_status-engines_field.data': 'fault_status',
    
    # MAVCTRL (Controle do Sistema)
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-path_dev_field.x': 'path_dev_x',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-path_dev_field.y': 'path_dev_y',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-path_dev_field.z': 'path_dev_z',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-rpy_field.x': 'ctrl_roll',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-rpy_field.y': 'ctrl_pitch',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavctrl-rpy_field.z': 'ctrl_yaw',

    # Bateria
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-battery_field.current': 'batt_current',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-battery_field.percentage': 'batt_pct',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-battery_field.voltage': 'batt_voltage',

    # IMU (Sensores Brutos)
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.angular_velocity.x': 'imu_gyro_x',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.angular_velocity.y': 'imu_gyro_y',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.angular_velocity.z': 'imu_gyro_z',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.linear_acceleration.x': 'imu_accel_x',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.linear_acceleration.y': 'imu_accel_y',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-imu-data_field.linear_acceleration.z': 'imu_accel_z',

    # VFR HUD (Dados de Voo consolidados)
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-vfr_hud_field.airspeed': 'hud_airspeed',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-vfr_hud_field.altitude': 'hud_altitude',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-vfr_hud_field.climb': 'hud_climb_rate',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-vfr_hud_field.throttle': 'hud_throttle',

    # Erros de Navegação (Cruciais para IA)
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-errors_field.aspd_error': 'err_airspeed',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-errors_field.alt_error': 'err_altitude',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-errors_field.xtrack_error': 'err_xtrack',

    # Nav Info (Comandado vs Medido)
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-pitch_field.commanded': 'pitch_cmd',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-pitch_field.measured': 'pitch_meas',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-roll_field.commanded': 'roll_cmd',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-nav_info-roll_field.measured': 'roll_meas',
    
    # Estimativa de Vento
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-wind_estimation_field.twist.linear.x': 'wind_x',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-wind_estimation_field.twist.linear.y': 'wind_y',
    'carbonZ_2018-07-18-15-53-31_1_engine_failure-mavros-wind_estimation_field.twist.linear.z': 'wind_z',
}

# Como são 105 colunas, para as que eu não mapeei manualmente acima, 
# podemos aplicar uma regra automática para limpar o prefixo:
def clean_column_name(col):
    if col in rename_dict:
        return rename_dict[col]
    # Remove o prefixo longo e campos genéricos como "_field" e ".data"
    new_name = col.split('-')[-1].replace('_field', '').replace('.data', '').replace('pose.pose.', '')
    return new_name

# Aplicando ao DataFrame
df.columns = [clean_column_name(c) for c in df.columns]

print("Novas colunas (exemplo):", df.columns[:10].tolist())

In [ ]:
df

# Visualizando as colunas 

In [ ]:
plot_flight_data_by_source(df)

## Eliminando colunas constantes

In [ ]:
const_cols = ['batt_current',
              'batt_pct',
              'battery.power_supply_status',
              'batt_voltage',
              'global.status.status',
              'local.twist.twist.angular.x',
              'local.twist.twist.angular.y',
              'local.twist.twist.angular.z',
              'target_global.yaw',
              'path_dev_x'
              ]


cols_degrau = ['imu_gyro_x',
               'imu_gyro_y',
               'imu_gyro_z',
               'imu_accel_x',
               'imu_accel_y',
               'imu_accel_z',
               'data.orientation.w',
               'data.orientation.x',
               'data.orientation.y',
               'data.orientation.z',
               'odom.orientation.w',
               'odom.orientation.x',
               'odom.orientation.y',
               'odom.orientation.z',
               'odom.twist.twist.angular.x',
               'odom.twist.twist.angular.y',
               'odom.twist.twist.angular.z',
               'orientation.w',
               'orientation.x',
               'orientation.y',
               'orientation.z',
               'velocity.twist.angular.x',
               'velocity.twist.angular.y',
               'velocity.twist.angular.z',
               'hud_climb_rate',
               'vfr_hud.heading',
               'hud_throttle',
               'ctrl_roll',
               'ctrl_pitch',
               'local.orientation.w',
               'local.orientation.x',
               'local.orientation.y',
               'local.orientation.z',
               ]

col_repetidas = ['target_global.velocity.x',
                 'target_global.velocity.y',
                 'local.velocity.x',
                 'local.velocity.y',
                 'odom.position.x',
                 'odom.position.y',
                 'odom.position.z',
                 'local.position.x',
                 'local.position.y'
                   ]

In [ ]:
df1 = df.drop(columns=const_cols)

print(f"Colunas removidas: {len(const_cols)}")
print(f"Colunas restantes no dataset: {df1.shape[1]}")

In [ ]:
df2 = df1.drop(columns=cols_degrau)

print(f"Colunas removidas: {len(cols_degrau)}")
print(f"Colunas restantes no dataset: {df2.shape[1]}")

In [ ]:
df3 = df2.drop(columns=col_repetidas)

print(f"Colunas removidas: {len(cols_degrau)}")
print(f"Colunas restantes no dataset: {df3.shape[1]}")

In [ ]:
plot_flight_data_by_source(df3)

In [ ]:
# 1. Transformar o tempo em um índice de tempo real (timedelta) para o Pandas entender a frequência
df['temp_index'] = pd.to_timedelta(df['%time'], unit='s')
df_resampled = df.set_index('temp_index')

# 2. Reamostrar para 10ms (0.01s)
# Usamos 'mean' para dados numéricos, mas como você já limpou o df, 
# o 'first' ou 'nearest' também funcionam bem.
df_100hz = df_resampled.resample('10L').first() # '10L' significa 10 milissegundos

# 3. Interpolação para evitar os zeros ou NaNs criados pelo resample
# Como aviões não se teletransportam, a interpolação linear é muito segura aqui
df_100hz = df_100hz.interpolate(method='linear')

# 4. Voltar o tempo para uma coluna numérica
df_100hz['%time'] = df_100hz.index.total_seconds()
df_100hz.reset_index(drop=True, inplace=True)

print(f"Novo shape a 100Hz: {df_100hz.shape}")
